In [1]:
import datasets
import duckdb
datasets.logging.set_verbosity_error() 

In [2]:
con = duckdb.connect()

In [4]:
from datasets import load_dataset

dataset = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_All_Beauty",
    trust_remote_code=True,
    split="full",
    cache_dir="../data/raw"
)

In [5]:
dataset[0]


{'rating': 5.0,
 'title': 'Such a lovely scent but not overpowering.',
 'text': "This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like you need a lot of it though to get the texture I want. I have a lot of hair, medium thickness. I am comparing to other brands with yucky chemicals so I'm gonna stick with this. Try it!",
 'images': [],
 'asin': 'B00YQ6X8EO',
 'parent_asin': 'B00YQ6X8EO',
 'user_id': 'AGKHLEW2SOWHNMFQIJGBECAF7INQ',
 'timestamp': 1588687728923,
 'helpful_vote': 0,
 'verified_purchase': True}

In [6]:
from datasets import load_dataset

dataset_meta = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_meta_All_Beauty",
    trust_remote_code=True,
    split="full",
    cache_dir="../data/raw"
)

raw/meta_categories/meta_All_Beauty.json(…):   0%|          | 0.00/213M [00:00<?, ?B/s]

Generating full split:   0%|          | 0/112590 [00:00<?, ? examples/s]

In [7]:
dataset_meta[0]

{'main_category': 'All Beauty',
 'title': 'Howard LC0008 Leather Conditioner, 8-Ounce (4-Pack)',
 'average_rating': 4.8,
 'rating_number': 10,
 'features': [],
 'description': [],
 'price': 'None',
 'images': {'hi_res': [None,
   'https://m.media-amazon.com/images/I/71i77AuI9xL._SL1500_.jpg'],
  'large': ['https://m.media-amazon.com/images/I/41qfjSfqNyL.jpg',
   'https://m.media-amazon.com/images/I/41w2yznfuZL.jpg'],
  'thumb': ['https://m.media-amazon.com/images/I/41qfjSfqNyL._SS40_.jpg',
   'https://m.media-amazon.com/images/I/41w2yznfuZL._SS40_.jpg'],
  'variant': ['MAIN', 'PT01']},
 'videos': {'title': [], 'url': [], 'user_id': []},
 'store': 'Howard Products',
 'categories': [],
 'details': '{"Package Dimensions": "7.1 x 5.5 x 3 inches; 2.38 Pounds", "UPC": "617390882781"}',
 'parent_asin': 'B01CUPMQZE',
 'bought_together': None,
 'subtitle': None,
 'author': None}

In [9]:
print(dataset.column_names)

['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase']


In [11]:
dataset.features

{'rating': Value(dtype='float64', id=None),
 'title': Value(dtype='string', id=None),
 'text': Value(dtype='string', id=None),
 'images': [{'attachment_type': Value(dtype='string', id=None),
   'large_image_url': Value(dtype='string', id=None),
   'medium_image_url': Value(dtype='string', id=None),
   'small_image_url': Value(dtype='string', id=None)}],
 'asin': Value(dtype='string', id=None),
 'parent_asin': Value(dtype='string', id=None),
 'user_id': Value(dtype='string', id=None),
 'timestamp': Value(dtype='int64', id=None),
 'helpful_vote': Value(dtype='int64', id=None),
 'verified_purchase': Value(dtype='bool', id=None)}

In [23]:
print("Number of rows in reviews dataset:", dataset.num_rows)
print("Number of columns in reviews dataset:", dataset.num_columns)
print("Number of rows in meta dataset:", dataset_meta.num_rows)
print("Number of columns in meta dataset:", dataset_meta.num_columns)

Number of rows in reviews dataset: 701528
Number of columns in reviews dataset: 10
Number of rows in meta dataset: 112590
Number of columns in meta dataset: 16


In [ ]:
arrow_tbl = dataset.data.table   

con.register("reviews", arrow_tbl)

In [17]:
result = con.execute("""
    SELECT
        rating,
        COUNT(*) AS n
    FROM reviews
    GROUP BY rating
    ORDER BY rating
""").fetch_df()

result

,rating,n
0,1.0,102080
1,2.0,43034
2,3.0,56307
3,4.0,79381
4,5.0,420726


In [19]:
head = con.execute("SELECT * FROM reviews LIMIT 5").fetch_df()
head

,rating,title,text,images,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase
0,5.0,Such a lovely scent but not overpowering.,This spray is really nice. It smells really go...,[],B00YQ6X8EO,B00YQ6X8EO,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588687728923,0,True
1,4.0,Works great but smells a little weird.,"This product does what I need it to do, I just...",[],B081TJ8YS3,B081TJ8YS3,AGKHLEW2SOWHNMFQIJGBECAF7INQ,1588615855070,1,True
2,5.0,Yes!,"Smells good, feels great!",[],B07PNNCSP9,B097R46CSY,AE74DYR3QUGVPZJ3P7RFWBGIX7XQ,1589665266052,2,True
3,1.0,Synthetic feeling,Felt synthetic,[],B09JS339BZ,B09JS339BZ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,1643393630220,0,True
4,5.0,A+,Love it,[],B08BZ63GMJ,B08BZ63GMJ,AFQLNQNQYFWQZPJQZS6V3NZU4QBQ,1609322563534,0,True


In [20]:
arrow_meta = dataset_meta.data.table   

con.register("meta", arrow_meta)

In [22]:
head_meta = con.execute("""
SELECT * FROM meta LIMIT 5
                   """).fetch_df()
head_meta

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,All Beauty,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,10,[],[],None,"{'hi_res': [None, 'https://m.media-amazon.com/...","{'title': [], 'url': [], 'user_id': []}",Howard Products,[],"{""Package Dimensions"": ""7.1 x 5.5 x 3 inches; ...",B01CUPMQZE,None,None,None
1,All Beauty,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,4.5,3,[],[],None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Yes To,[],"{""Item Form"": ""Powder"", ""Skin Type"": ""Acne Pro...",B076WQZGPM,None,None,None
2,All Beauty,Eye Patch Black Adult with Tie Band (6 Per Pack),4.4,26,[],[],None,"{'hi_res': [None, None], 'large': ['https://m....","{'title': [], 'url': [], 'user_id': []}",Levine Health Products,[],"{""Manufacturer"": ""Levine Health Products""}",B000B658RI,None,None,None
3,All Beauty,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",3.1,102,[],[],None,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': [], 'url': [], 'user_id': []}",Cherioll,[],"{""Brand"": ""Cherioll"", ""Item Form"": ""Powder"", ""...",B088FKY3VD,None,None,None
4,All Beauty,Precision Plunger Bars for Cartridge Grips – 9...,4.3,7,"[Material: 304 Stainless Steel; Brass tip, Len...",[The Precision Plunger Bars are designed to wo...,None,"{'hi_res': [None], 'large': ['https://m.media-...","{'title': [], 'url': [], 'user_id': []}",Precision,[],"{""UPC"": ""644287689178""}",B07NGFDN6G,None,None,None


### Merging both Meta and Review Files

In [27]:
dataset.column_names

['rating',
 'title',
 'text',
 'images',
 'asin',
 'parent_asin',
 'user_id',
 'timestamp',
 'helpful_vote',
 'verified_purchase']

In [ ]:
result = con.execute("""
    COPY(
        SELECT
            r.rating, r.title, r.text, r.verified_purchase, m.title AS product_title, m.average_rating,m.price, m.description, m.store, m.details
        FROM reviews r
        JOIN meta m ON r.parent_asin = m.parent_asin)
                     
""").fetch_df()
result.head()

,rating,title,text,verified_purchase,product_title,average_rating,price,description,store,details
0,4.0,Keeps leather from drying out,I have a leather couch and because we live in ...,True,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",4.8,None,[],Howard Products,"{""Package Dimensions"": ""7.1 x 5.5 x 3 inches; ..."
1,5.0,Five Stars,Grandson has had very good results with this p...,True,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,4.5,None,[],Yes To,"{""Item Form"": ""Powder"", ""Skin Type"": ""Acne Pro..."
2,5.0,Good Eye Patch,It is good. No problems. Eye patch is as adver...,True,Eye Patch Black Adult with Tie Band (6 Per Pack),4.4,None,[],Levine Health Products,"{""Manufacturer"": ""Levine Health Products""}"
3,5.0,Wonderful and super realistic,I have trichotillomania which is a hair pullin...,True,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",3.1,None,[],Cherioll,"{""Brand"": ""Cherioll"", ""Item Form"": ""Powder"", ""..."
4,5.0,Missing Review.......... It’s lost at Amazon S...,Where did my review go? I used your take a pic...,True,Precision Plunger Bars for Cartridge Grips – 9...,4.3,None,[The Precision Plunger Bars are designed to wo...,Precision,"{""UPC"": ""644287689178""}"


In [41]:
result.info()

<class 'pandas.DataFrame'>
RangeIndex: 701528 entries, 0 to 701527
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   rating             701528 non-null  float64
 1   title              701528 non-null  str    
 2   text               701528 non-null  str    
 3   verified_purchase  701528 non-null  bool   
 4   product_title      701528 non-null  str    
 5   average_rating     701528 non-null  float64
 6   price              701528 non-null  str    
 7   description        701528 non-null  object 
 8   store              651636 non-null  str    
 9   details            701528 non-null  str    
dtypes: bool(1), float64(2), object(1), str(6)
memory usage: 414.8+ MB


## Saving merged data locally as a parquet file

In [42]:
import os

data_dir = "../data/merged"
os.makedirs(data_dir, exist_ok=True)

result.to_parquet(os.path.join(data_dir, "merged_reviews_meta.parquet"), index=False)